# Сегментация дорожных знаков

Обучить DeepLabV3 ResNet50 выделять дорожные знаки и сравнить предсказания с эталонными масками.

[Описание и данные](README.md) · [История эксперимента](results/recorded_metrics.json)

**Запуск:** Python 3.11 или 3.12. Полное обучение требует данных и подходящего GPU. Выводы очищены; численные результаты ниже появятся после выполнения.

## Подготовка окружения

Ячейка устанавливает зависимости проекта. Локально рекомендуется отдельное виртуальное окружение.

In [ ]:
%pip -q install -r https://raw.githubusercontent.com/Xokage9/ml-portfolio/main/projects/04_segmentation/requirements.txt


## Конфигурация и пути

In [ ]:
from __future__ import annotations

import json
import random
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "sign_dataset"
TEST_IMAGES_DIR = PROJECT_ROOT / "test_images_lab6"
REPORT_DIR = PROJECT_ROOT / "reports" / "lab6_torchvision"
VIS_DIR = REPORT_DIR / "visualizations"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

IMG_TRAIN_DIR = DATA_ROOT / "images" / "train"
IMG_VAL_DIR = DATA_ROOT / "images" / "val"
LBL_TRAIN_DIR = DATA_ROOT / "labels" / "train"
LBL_VAL_DIR = DATA_ROOT / "labels" / "val"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python root:", PROJECT_ROOT)
print("Device:", device)
print("Dataset exists:", DATA_ROOT.exists())
print("Test images exists:", TEST_IMAGES_DIR.exists())


## Конвертация исходных масок

In [ ]:
import json
from pathlib import Path
import numpy as np
import cv2

DATA_ROOT = Path("sign_dataset")
for split in ["train", "val"]:
    img_dir = DATA_ROOT / "images" / split
    lbl_dir = DATA_ROOT / "labels" / split
    lbl_dir.mkdir(parents=True, exist_ok=True)

    written, non_empty, empty = 0, 0, 0
    for jp in sorted(img_dir.glob("*_coco.json")):
        d = json.loads(jp.read_text(encoding="utf-8"))

        class_ids = d.get("class_ids", [])
        masks = np.array(d.get("masks", []), dtype=np.uint8)
        if masks.size == 0 or len(class_ids) == 0:
            continue


        if masks.ndim == 3 and masks.shape[2] == len(class_ids):
            masks_nhw = np.moveaxis(masks, 2, 0)
        elif masks.ndim == 3 and masks.shape[0] == len(class_ids):
            masks_nhw = masks
        else:
            continue

        n, h, w = masks_nhw.shape
        lines = []

        for i in range(n):
            cls = int(class_ids[i])
            m = (masks_nhw[i] > 0).astype(np.uint8)
            cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            for c in cnts:
                if cv2.contourArea(c) < 4:
                    continue
                pts = c.squeeze(1)
                if pts.ndim != 2 or len(pts) < 3:
                    continue

                vals = []
                for x, y in pts:
                    vals.append(f"{x / max(w - 1, 1):.6f}")
                    vals.append(f"{y / max(h - 1, 1):.6f}")
                lines.append(f"{cls} " + " ".join(vals))

        image_name = jp.name.replace("_coco.json", "")
        out_name = f"{Path(image_name).stem}.txt"
        out_path = lbl_dir / out_name
        out_path.write_text(("\n".join(lines) + "\n") if lines else "", encoding="utf-8")

        written += 1
        if lines:
            non_empty += 1
        else:
            empty += 1

    print(split, {"written": written, "non_empty": non_empty, "empty": empty})


## Выбор классов по обучающей части

In [ ]:
def parse_yolo_poly_line(line: str):
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    cls_id = int(float(parts[0]))
    coords = [float(x) for x in parts[1:]]
    if len(coords) % 2 != 0:
        return None
    pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
    return cls_id, pts


def class_frequency(label_dir: Path):
    freq = Counter()
    for txt in sorted(label_dir.glob("*.txt")):
        with txt.open("r", encoding="utf-8") as f:
            for line in f:
                parsed = parse_yolo_poly_line(line)
                if parsed is None:
                    continue
                cls, _ = parsed
                freq[cls] += 1
    return freq


mapping_path = PROJECT_ROOT / "reports" / "lab6" / "class_mapping.json"
if mapping_path.exists():
    mapping_data = json.loads(mapping_path.read_text(encoding="utf-8"))
    selected_raw_classes = [int(x) for x in mapping_data["selected_raw_classes"]]
else:
    freq = class_frequency(LBL_TRAIN_DIR)
    selected_raw_classes = [c for c, _ in freq.most_common(8)]

raw_to_model = {raw: i + 1 for i, raw in enumerate(selected_raw_classes)}
id_to_name = {0: "background"}
for raw, idx in raw_to_model.items():
    id_to_name[idx] = f"class_{raw}"

if not raw_to_model:
    raise ValueError("Не найдены размеченные классы в sign_dataset/labels/train")
num_classes = int(max(raw_to_model.values())) + 1
print("Selected raw classes:", selected_raw_classes)
print("Model classes:", id_to_name)
print("num_classes:", num_classes)


## Dataset и DataLoader

In [ ]:


BATCH_SIZE = 2
ACCUM_STEPS = 2              
NUM_WORKERS = 0 
class RoadSignsSegDataset(Dataset):
    def __init__(self, images_dir: Path, labels_dir: Path, raw_to_model: dict[int, int], image_size: int = 512):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.raw_to_model = raw_to_model
        self.image_paths = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        self.image_size = image_size
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def _build_mask(self, image_w: int, image_h: int, label_path: Path):
        mask = np.zeros((image_h, image_w), dtype=np.uint8)
        if not label_path.exists():
            return mask

        with label_path.open("r", encoding="utf-8") as f:
            for line in f:
                parsed = parse_yolo_poly_line(line)
                if parsed is None:
                    continue
                raw_cls, pts_norm = parsed
                if raw_cls not in self.raw_to_model:
                    continue
                pts = pts_norm.copy()
                pts[:, 0] = np.clip(pts[:, 0] * image_w, 0, image_w - 1)
                pts[:, 1] = np.clip(pts[:, 1] * image_h, 0, image_h - 1)
                pts = pts.astype(np.int32)
                if len(pts) >= 3:
                    cv2.fillPoly(mask, [pts], color=int(self.raw_to_model[raw_cls]))
        return mask

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        lbl_path = self.labels_dir / f"{img_path.stem}.txt"

        with Image.open(img_path) as im:
            img = im.convert("RGB")

        w, h = img.size
        mask = self._build_mask(w, h, lbl_path)

        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = cv2.resize(mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        img_t = self.to_tensor(img)
        mask_t = torch.from_numpy(mask).long()
        return img_t, mask_t, img_path.name


train_ds = RoadSignsSegDataset(IMG_TRAIN_DIR, LBL_TRAIN_DIR, raw_to_model, image_size=512)
val_ds = RoadSignsSegDataset(IMG_VAL_DIR, LBL_VAL_DIR, raw_to_model, image_size=512)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

print("Train images:", len(train_ds))
print("Val images:", len(val_ds))


## Модель и оптимизатор

In [ ]:
model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
if model.aux_classifier is not None:
    model.aux_classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 12
print("Model ready. EPOCHS:", EPOCHS)


## Проверка масок validation

In [ ]:
non_empty = 0
for _, m, _ in val_ds:
    non_empty += int((m > 0).any())
print("non-empty val masks:", non_empty, "/", len(val_ds))


## Обучение и выбор checkpoint

In [ ]:



use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
def batch_binary_iou(pred_cls, gt_cls):
    pred_fg = (pred_cls > 0)
    gt_fg = (gt_cls > 0)

    inter = (pred_fg & gt_fg).flatten(1).sum(1).float()
    union = (pred_fg | gt_fg).flatten(1).sum(1).float()

    iou = torch.where(union > 0, inter / union, torch.ones_like(union))
    return iou.mean().item()

def train_one_epoch(model, loader, optimizer, criterion, device, scaler, accum_steps=1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (imgs, masks, _) in enumerate(tqdm(loader, leave=False), start=1):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(imgs)["out"]
            loss = criterion(out, masks) / accum_steps

        scaler.scale(loss).backward()

        if step % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * accum_steps * imgs.size(0)

    if len(loader) % accum_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_iou = 0.0
    n_batches = 0

    for imgs, masks, _ in tqdm(loader, leave=False):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(imgs)["out"]
            loss = criterion(out, masks)

        preds = torch.argmax(out, dim=1)
        iou = batch_binary_iou(preds, masks)

        total_loss += loss.item() * imgs.size(0)
        total_iou += iou * imgs.size(0)
        n_batches += imgs.size(0)

    val_loss = total_loss / len(loader.dataset)
    val_iou = total_iou / max(n_batches, 1)
    return val_loss, val_iou

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

history = []
best_val_iou = -1.0
best_path = REPORT_DIR / "best_deeplabv3_sign8.pth"

for epoch in range(1, EPOCHS + 1):
    if use_amp:
        torch.cuda.reset_peak_memory_stats()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler, accum_steps=ACCUM_STEPS)
    val_loss, val_iou = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_iou)

    lr = optimizer.param_groups[0]["lr"]
    peak_vram = (torch.cuda.max_memory_allocated() / 1024**3) if use_amp else 0.0

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_iou": val_iou,
        "lr": lr,
        "peak_vram_gb": peak_vram,
    })

    print(f"Epoch {epoch:02d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
          f"val_iou={val_iou:.4f}, lr={lr:.2e}, peak_vram={peak_vram:.2f}GB")

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), best_path)

    if use_amp:
        torch.cuda.empty_cache()

pd.DataFrame(history).to_csv(REPORT_DIR / "train_history.csv", index=False)
print("Best model:", best_path, "| best_val_iou:", round(best_val_iou, 4))



## Бинарные метрики foreground

In [ ]:
best_path = REPORT_DIR / "best_deeplabv3_sign8.pth"

if best_path.exists():
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    print("Loaded:", best_path)
else:
    print("Checkpoint not found:", best_path)

model.eval()

def binary_metrics(pred_mask: np.ndarray, gt_mask: np.ndarray):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, np.logical_not(gt)).sum()
    fn = np.logical_and(np.logical_not(pred), gt).sum()
    union = np.logical_or(pred, gt).sum()

    iou = float(tp / union) if union > 0 else 1.0
    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else 1.0
    recall = float(tp / (tp + fn)) if (tp + fn) > 0 else 1.0
    l2 = float(np.sqrt(np.mean((pred.astype(np.float32) - gt.astype(np.float32)) ** 2)))
    return iou, precision, recall, l2


rows = []
with torch.no_grad():
    for imgs, masks, names in tqdm(val_loader):
        logits = model(imgs.to(device))["out"]
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        gts = masks.numpy()

        for i in range(len(names)):
            pred_bin = (preds[i] > 0).astype(np.uint8)
            gt_bin = (gts[i] > 0).astype(np.uint8)
            iou, precision, recall, l2 = binary_metrics(pred_bin, gt_bin)
            rows.append({
                "image": names[i],
                "iou": iou,
                "precision": precision,
                "recall": recall,
                "l2": l2,
            })

val_metrics_df = pd.DataFrame(rows)
val_summary = {
    "mean_iou": float(val_metrics_df["iou"].mean()),
    "mean_precision": float(val_metrics_df["precision"].mean()),
    "mean_recall": float(val_metrics_df["recall"].mean()),
    "mean_l2": float(val_metrics_df["l2"].mean()),
    "pct_iou_ge_05": float((val_metrics_df["iou"] >= 0.5).mean() * 100.0),
    "pct_iou_ge_075": float((val_metrics_df["iou"] >= 0.75).mean() * 100.0),
    "pct_iou_ge_09": float((val_metrics_df["iou"] >= 0.9).mean() * 100.0),
}

val_metrics_df.to_csv(REPORT_DIR / "val_per_image_metrics.csv", index=False)
pd.DataFrame([val_summary]).to_csv(REPORT_DIR / "val_summary_metrics.csv", index=False)
val_summary


## Необязательные визуальные примеры

In [ ]:
def overlay_prediction(image_rgb: np.ndarray, pred_mask: np.ndarray):
    color = np.zeros_like(image_rgb)
    color[pred_mask > 0] = np.array([255, 80, 80], dtype=np.uint8)
    out = cv2.addWeighted(image_rgb, 0.75, color, 0.25, 0)
    return out


test_image_paths = sorted([p for p in TEST_IMAGES_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:17]
print("Selected test images:", len(test_image_paths))
for p in test_image_paths:
    print(" -", p.name)

model.eval()
test_rows = []
for p in test_image_paths:
    img = Image.open(p).convert("RGB")
    orig = np.array(img)
    img_resized = img.resize((512, 512), Image.BILINEAR)
    inp = transforms.ToTensor()(img_resized).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inp)["out"]
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    pred_orig_size = cv2.resize(pred, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_NEAREST)
    vis = overlay_prediction(orig, pred_orig_size)
    out_path = VIS_DIR / f"pred_{p.stem}.png"
    Image.fromarray(vis).save(out_path)
    test_rows.append({"image": p.name, "visualization": str(out_path)})

pd.DataFrame(test_rows).to_csv(REPORT_DIR / "test10_visualizations.csv", index=False)
print("Saved visualizations to:", VIS_DIR)


## Как читать результат

Модель предсказывает восемь классов и фон. Итоговый отчёт исходного эксперимента объединяет все знаки в foreground и считает бинарные метрики; mean foreground IoU нельзя называть многоклассовым mIoU.

Это validation, используемая при выборе checkpoint. Фотографии без ручных масок дают только визуальные примеры. После исправления обработки пустых масок и усреднения оценку необходимо пересчитать.

Сохранённая история исходного эксперимента находится отдельно в `results/recorded_metrics.json`. Полное GPU-обучение при подготовке этой версии не выполнялось.